# MASTER JOINER

Done and missing:

- [x] Merge 1
- [x] Merge 2
- [] Merge 3

In [1]:
# === IMPORTS ===
import pandas as pd
from pathlib import Path

# ==== Define paths ====
data_root = "../Data/"
data_folder = "../Data/gcp_order/dtu_findit/extraction_and_processing/"

# ==== Define file name ====
file_exstracted_metrics = "extracted_metrics.csv"       # containing the extracted metrics from the PDFs (e.g., number of pages, words, etc.)
file_handin_month = "handin_month_summary.csv"          # containing the hand-in month for each thesis, used for seasonality analysis
file_fig_tab_ref = "extracted_metrics_fig-tab-ref.csv"  # containing the number of figures, tables, and references extracted from the PDFs
file_lingustics = "linguistics_exstract.csv"            # containing the linguistic features extracted from the PDFs (e.g., readability scores, average sentence length, etc.)

In [2]:
# ==== FUNCTION ====
def load_csv_to_df(csv_path, sep=";"):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        print(f"Successfully loaded CSV from {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

In [3]:
# ==== LOAD ====
csv_path_extracted_metrics = Path(data_root) / file_exstracted_metrics
df_extracted_metrics = load_csv_to_df(csv_path_extracted_metrics, sep=";")

csv_path_handin = Path(data_folder) / file_handin_month
df_handin_month = load_csv_to_df(csv_path_handin, sep=";")

csv_path_extracted_metrics_fig_tab_ref = Path(data_root) / file_fig_tab_ref
df_extracted_metrics_fig_tab_ref = load_csv_to_df(csv_path_extracted_metrics_fig_tab_ref, sep=",")

Successfully loaded CSV from ../Data/extracted_metrics.csv
DataFrame shape: (6302, 7)
DataFrame columns: ['pdf_file', 'member_id_ss_metrics', 'num_tot_pages', 'num_cont_pages', 'match_trigger', 'num_words_full', 'num_words_cont']

Successfully loaded CSV from ../Data/gcp_order/dtu_findit/extraction_and_processing/handin_month_summary.csv
DataFrame shape: (6332, 3)
DataFrame columns: ['filename', 'handin_month', 'corrupt_cid']

Successfully loaded CSV from ../Data/extracted_metrics_fig-tab-ref.csv
DataFrame shape: (6332, 4)
DataFrame columns: ['pdf_file', 'num_figures', 'num_tables', 'num_references']



In [4]:
# ==== MERGE 1 ====
# df_handin_month
# df_extracted_metrics

# join the two DataFrames on the "pdf_file" column
merged_metrics_handin = pd.merge(
    df_handin_month,
    df_extracted_metrics,
    left_on="filename",
    right_on="pdf_file",
    how="left",
)

# drop column "corrupt_cid"
merged_metrics_handin = merged_metrics_handin.drop(columns=["corrupt_cid"], errors="ignore")
# drop column "filename"
merged_metrics_handin = merged_metrics_handin.drop(columns=["filename"], errors="ignore")
# drop column "match_trigger"
merged_metrics_handin = merged_metrics_handin.drop(columns=["match_trigger"], errors="ignore")

# move column "handin_month" to the end of the DataFrame
handin_month_col = merged_metrics_handin.pop("handin_month")
merged_metrics_handin["handin_month"] = handin_month_col


print("Merged DataFrame shape:", merged_metrics_handin.shape)
display(merged_metrics_handin.head())

Merged DataFrame shape: (6332, 7)


,pdf_file,member_id_ss_metrics,num_tot_pages,num_cont_pages,num_words_full,num_words_cont,handin_month
0,5c7a72fdd9001d361b4cd3e4_Coronary atherosclero...,5c7a72fdd9001d361b4cd3e4,118.0,108.0,33336.0,31691.0,February 2019
1,5ce67fe9d9001d016b4fccc7_Experimental analysis...,5ce67fe9d9001d016b4fccc7,66.0,62.0,15415.0,13882.0,April 2019
2,5cee68eed9001d2064299318_Deep Reinforcement Le...,5cee68eed9001d2064299318,61.0,51.0,20540.0,17924.0,April 2019
3,5cee68fbd9001d206429932e_Analysis of pyrolysis...,5cee68fbd9001d206429932e,76.0,56.0,20652.0,13616.0,April 2019
4,5ce52e6ad9001d016b1cc373_Production line using...,5ce52e6ad9001d016b1cc373,60.0,58.0,9811.0,9713.0,October 2022


In [5]:
# ==== MERGE 2 ====
# merged_metrics_handin
# df_extracted_metrics_fig_tab_ref

# join the two DataFrames on the "pdf_file" column
merged_metrics_handin_figures_tables_references = pd.merge(
    merged_metrics_handin,
    df_extracted_metrics_fig_tab_ref,
    left_on="pdf_file",
    right_on="pdf_file",
    how="left",
)

print("Merged DataFrame shape:", merged_metrics_handin_figures_tables_references.shape)
display(merged_metrics_handin_figures_tables_references.head())

Merged DataFrame shape: (6332, 10)


,pdf_file,member_id_ss_metrics,num_tot_pages,num_cont_pages,num_words_full,num_words_cont,handin_month,num_figures,num_tables,num_references
0,5c7a72fdd9001d361b4cd3e4_Coronary atherosclero...,5c7a72fdd9001d361b4cd3e4,118.0,108.0,33336.0,31691.0,February 2019,49.0,13.0,45.0
1,5ce67fe9d9001d016b4fccc7_Experimental analysis...,5ce67fe9d9001d016b4fccc7,66.0,62.0,15415.0,13882.0,April 2019,21.0,2.0,53.0
2,5cee68eed9001d2064299318_Deep Reinforcement Le...,5cee68eed9001d2064299318,61.0,51.0,20540.0,17924.0,April 2019,20.0,12.0,67.0
3,5cee68fbd9001d206429932e_Analysis of pyrolysis...,5cee68fbd9001d206429932e,76.0,56.0,20652.0,13616.0,April 2019,35.0,29.0,27.0
4,5ce52e6ad9001d016b1cc373_Production line using...,5ce52e6ad9001d016b1cc373,60.0,58.0,9811.0,9713.0,October 2022,33.0,2.0,7.0


### export `.parquet`

In [6]:
export = input("Export merged DataFrame to Parquet? (y/N): ").strip().lower()

if export == "y":
    pd.DataFrame.to_parquet(
        merged_metrics_handin_figures_tables_references,
        Path(data_folder) / "master_thesis_metrics_handin_figures_tables_references.parquet",
    )
    print(f"Merged {merged_metrics_handin_figures_tables_references.shape[0]} rows exported to Parquet successfully.")
    print(f"Exported file path: {Path(data_folder) / 'master_thesis_metrics_handin_figures_tables_references.parquet'}")
else:
    print("Export skipped.")

Merged 6332 rows exported to Parquet successfully.
Exported file path: ../Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_handin_figures_tables_references.parquet


# JOIN WHEN lingustics is done

In [ ]:
# ==== MERGE 3 ====

# load the linguistics DataFrame
csv_path_lingustics = Path(data_folder) / file_lingustics
df_lingustics = load_csv_to_df(csv_path_lingustics, sep=",")

# load the previously merged DataFrame (.parquet)
master_thesis_metrics_handin_figures_tables_references = pd.read_parquet(Path(data_folder) / "master_thesis_metrics_handin_figures_tables_references.parquet")


# master_thesis_metrics_handin_figures_tables_references
# df_lingustics

# join the two DataFrames on the "pdf_file" column
merged_metrics_handin_figures_tables_references_lingustics = pd.merge(
    master_thesis_metrics_handin_figures_tables_references,
    df_lingustics,
    left_on="pdf_file",
    right_on="pdf_file",
    how="left",
)

print("Merged DataFrame shape:", merged_metrics_handin_figures_tables_references_lingustics.shape)
display(merged_metrics_handin_figures_tables_references_lingustics.head())

In [ ]:
export = input("Export merged DataFrame to Parquet? (y/N): ").strip().lower()

if export == "y":
    pd.DataFrame.to_parquet(
        merged_metrics_handin_figures_tables_references,
        Path(data_folder) / "master_thesis_metrics_handin_figures_tables_references.parquet",
    )
    print(f"Merged {merged_metrics_handin_figures_tables_references.shape[0]} rows exported to Parquet successfully.")
    print(f"Exported file path: {Path(data_folder) / 'master_thesis_metrics_handin_figures_tables_references.parquet'}")
else:
    print("Export skipped.")